# md2LLM - Fine-tune your personal model

This notebook fine-tunes a small language model on your personal notes.

**Before running:**
1. Make sure GPU is enabled: Runtime -> Change runtime type -> T4 GPU
2. Run all cells: Runtime -> Run all
3. Upload your `training_data.jsonl` when Cell 2 prompts you

**Time:** ~15-20 minutes on free T4 GPU

In [ ]:
# Cell 1 - Install dependencies
print('Installing dependencies...')
!pip install unsloth transformers trl datasets peft accelerate bitsandbytes -q
print('Done!')

In [ ]:
# Cell 2 - Upload your training data
from google.colab import files
import json

print('Please upload your training_data.jsonl file from md2LLM')
uploaded = files.upload()
data_file = list(uploaded.keys())[0]

pairs = []
with open(data_file) as f:
    for line in f:
        line = line.strip()
        if line:
            pairs.append(json.loads(line))

print(f'Loaded {len(pairs)} training pairs from {data_file}')

In [ ]:
# Cell 3 - Configure training
# Edit these values if you want to change the model or settings

MODEL_NAME  = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_NAME = 'my-md2llm-model'
EPOCHS      = 3
LORA_RANK   = 16

print(f'Model:  {MODEL_NAME}')
print(f'Output: {OUTPUT_NAME}')
print(f'Pairs:  {len(pairs)}')
print(f'Epochs: {EPOCHS}')

In [ ]:
# Cell 4 - Load model with Unsloth
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME}...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

print('Model loaded and LoRA applied')

In [ ]:
# Cell 5 - Prepare dataset
from datasets import Dataset

def format_pair(pair):
    instruction = pair.get('instruction', '')
    output = pair.get('output', '')
    messages = pair.get('messages', [])

    if messages:
        result = ''
        for msg in messages:
            result += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
        return result
    return f'### Instruction:\n{instruction}\n\n### Response:\n{output}'

texts = [format_pair(p) for p in pairs]
dataset = Dataset.from_dict({'text': texts})

print(f'Dataset prepared: {len(texts)} examples')

In [ ]:
# Cell 6 - Train
from trl import SFTTrainer
from transformers import TrainingArguments

steps_per_epoch = max(1, len(texts) // 8)
total_steps = steps_per_epoch * EPOCHS

print(f'Training: {total_steps} total steps')
print('This takes 15-20 minutes on T4 GPU...')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir='./checkpoints',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        logging_steps=max(1, total_steps // 20),
        fp16=True,
        optim='adamw_8bit',
        seed=42,
        report_to='none',
    ),
)

trainer.train()
print('Training complete!')

In [ ]:
# Cell 7 - Save as GGUF
import os

print('Converting to GGUF format...')
print('This takes 2-3 minutes...')

model.save_pretrained_gguf(
    OUTPUT_NAME,
    tokenizer,
    quantization_method='q4_k_m',
)

gguf_files = []
for root, dirs, files_list in os.walk(OUTPUT_NAME):
    for f in files_list:
        if f.endswith('.gguf'):
            gguf_files.append(os.path.join(root, f))

if gguf_files:
    print(f'GGUF saved: {gguf_files[0]}')
else:
    print('Looking for GGUF in current directory...')
    gguf_files = [f for f in os.listdir('.') if f.endswith('.gguf')]
    print(f'Found: {gguf_files}')

In [ ]:
# Cell 8 - Download your model
from google.colab import files
import os

gguf_path = None
for root, dirs, files_list in os.walk('.'):
    for f in files_list:
        if f.endswith('.gguf'):
            gguf_path = os.path.join(root, f)
            break

if gguf_path:
    size_mb = os.path.getsize(gguf_path) / 1024 / 1024
    print(f'Downloading {gguf_path} ({size_mb:.0f} MB)...')
    files.download(gguf_path)
    print()
    print('Done! Next steps:')
    print('1. Move the downloaded .gguf file to your md2LLM models/ folder')
    print('2. Run: ollama create my-model -f training/Modelfile')
    print('3. Open md2LLM and go to chat')
else:
    print('GGUF file not found. Check the Files panel on the left.')